* [#41](https://github.com/salgo60/ifkdb/issues/41)
* denna Notebook [41_fotbolltransfers.ipynb](https://github.com/salgo60/ifkdb/blob/main/Notebook/41_fotbolltransfers.ipynb)

* Mixandmatch [catalog/7732](https://mix-n-match.toolforge.org/#/catalog/7732)

In [1]:
import sys
print(sys.executable)

/Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12


In [2]:
#!pip3 install aiohttp

In [3]:
import time

from datetime import datetime

now = datetime.now()
timestamp = now.timestamp()

start_time = time.time()
print("Start:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Start: 2026-03-20 14:09:58


In [41]:
import requests
from bs4 import BeautifulSoup
import time
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import json
import os
import re

BASE_PLAYER_URL = "https://fotbolltransfers.com/spelare/x/{}"

SAVE_FILE = "resultsPlayer_partial.json"
FINAL_FILE = "resultsPlayer.json"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; Wikidatascraper/1.0 salgo60@msn.com)"
}

session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
)

adapter = HTTPAdapter(max_retries=retries)
session.mount("https://", adapter)
session.mount("http://", adapter)

# resume
if os.path.exists(SAVE_FILE):
    with open(SAVE_FILE, "r", encoding="utf-8") as f:
        resultsPlayer = json.load(f)
    print(f"Återupptar från {len(resultsPlayer)} spelare")
else:
    resultsPlayer = []

existing_ids = {item["numeric_id"] for item in resultsPlayer}
start_id = max(existing_ids) + 1 if existing_ids else 1

print(f"Startar från player_id = {start_id}")

SAVE_EVERY = 100
MAX_EMPTY_STREAK = 200
empty_streak = 0


def slugify(text):
    text = text.lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = text.replace(" ", "-")
    return text


def clean_text(text):
    if not text:
        return None
    return re.sub(r"\s+", " ", text).strip()


def extract_profile_fields(soup):
    """
    Hämtar spelardata från tabellen med etikett i första <td>
    och värde i andra <td>.
    """
    wanted = {
        "Födelsedatum": "birth_date",
        "Längd": "height",
        "Vikt": "weight",
        "Position": "position",
        "Moderklubb": "youth_club",
        "Nationalitet": "nationality",
    }

    data = {v: None for v in wanted.values()}

    for row in soup.select("tbody tr"):
        cols = row.find_all("td")
        if len(cols) < 2:
            continue

        label = clean_text(cols[0].get_text(" ", strip=True))
        value = clean_text(cols[1].get_text(" ", strip=True))

        if not label:
            continue

        label = label.rstrip(":").strip()

        if label in wanted:
            data[wanted[label]] = value

    return data


#for player_id in tqdm(range(start_id, 15500)):
for player_id in tqdm(range(start_id, 200)):
    url = BASE_PLAYER_URL.format(player_id)

    try:
        r = session.get(url, headers=headers, timeout=10)

        if r.status_code != 200:
            empty_streak += 1
            continue

        soup = BeautifulSoup(r.text, "html.parser")

        if not soup.title or not soup.title.string:
            empty_streak += 1
            continue

        name = soup.title.string.split(" - ")[0].strip()

        if name == "Fotbolltransfers.com":
            empty_streak += 1
            continue

        slug = slugify(name)
        full_id = f"spelare/{slug}/{player_id}"
        full_url = f"https://fotbolltransfers.com/{full_id}"

        profile_data = extract_profile_fields(soup)

        resultsPlayer.append({
            "id": full_id,
            "name": name,
            "url": full_url,
            "numeric_id": player_id,
            "birth_date": profile_data["birth_date"],
            "height": profile_data["height"],
            "weight": profile_data["weight"],
            "position": profile_data["position"],
            "youth_club": profile_data["youth_club"],
            "nationality": profile_data["nationality"],
        })

        empty_streak = 0

    except requests.exceptions.RequestException as e:
        print(f"Error {player_id}: {e}")
        empty_streak += 1
        continue

    if player_id % SAVE_EVERY == 0:
        with open(SAVE_FILE, "w", encoding="utf-8") as f:
            json.dump(resultsPlayer, f, ensure_ascii=False)

    if empty_streak >= MAX_EMPTY_STREAK:
        print(f"\n🛑 Stoppar: {empty_streak} tomma i rad vid ID {player_id}")
        break

    time.sleep(0.2)

with open(FINAL_FILE, "w", encoding="utf-8") as f:
    json.dump(resultsPlayer, f, ensure_ascii=False, indent=2)

print(f"Klar! Spelare funna: {len(resultsPlayer)}")

Startar från player_id = 1


 31%|█████████████                             | 62/199 [01:26<19:48,  8.68s/it]

Error 62: HTTPSConnectionPool(host='fotbolltransfers.com', port=443): Max retries exceeded with url: /spelare/x/62 (Caused by ResponseError('too many 500 error responses'))


100%|█████████████████████████████████████████| 199/199 [03:50<00:00,  1.16s/it]

Klar! Spelare funna: 152


In [39]:
resultsPlayer

[{'id': 'spelare/kim-christensen/37',
  'name': 'Kim Christensen',
  'url': 'https://fotbolltransfers.com/spelare/kim-christensen/37',
  'numeric_id': 37},
 {'id': 'spelare/marcus-sandberg/38',
  'name': 'Marcus Sandberg',
  'url': 'https://fotbolltransfers.com/spelare/marcus-sandberg/38',
  'numeric_id': 38},
 {'id': 'spelare/karl-svensson/39',
  'name': 'Karl Svensson',
  'url': 'https://fotbolltransfers.com/spelare/karl-svensson/39',
  'numeric_id': 39},
 {'id': 'spelare/nicklas-carlsson/40',
  'name': 'Nicklas Carlsson',
  'url': 'https://fotbolltransfers.com/spelare/nicklas-carlsson/40',
  'numeric_id': 40},
 {'id': 'spelare/petter-björlund/41',
  'name': 'Petter Björlund',
  'url': 'https://fotbolltransfers.com/spelare/petter-björlund/41',
  'numeric_id': 41},
 {'id': 'spelare/tuomo-turunen/42',
  'name': 'Tuomo Turunen',
  'url': 'https://fotbolltransfers.com/spelare/tuomo-turunen/42',
  'numeric_id': 42},
 {'id': 'spelare/adam-johansson/43',
  'name': 'Adam Johansson',
  'url':

In [5]:
import re

BASE_TEAM_URL = "https://fotbolltransfers.com/klubbar/x/xx/{}"

SAVE_FILE = "resultsTeam_partial.json"
FINAL_FILE = "resultsTeam.json"

# 🔄 resume
if os.path.exists(SAVE_FILE):
    with open(SAVE_FILE, "r", encoding="utf-8") as f:
        resultsTeam = json.load(f)
else:
    resultsTeam = []

existing_ids = {item["numeric_id"] for item in resultsTeam}
start_id = max(existing_ids) + 1 if existing_ids else 1

def slugify(text):
    text = text.lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = text.replace(" ", "-")
    return text

for team_id in tqdm(range(start_id, 6800)):
    url = BASE_TEAM_URL.format(team_id)

    try:
        r = session.get(url, headers=headers, timeout=10)

        if r.status_code != 200:
            empty_streak += 1
            continue

        soup = BeautifulSoup(r.text, "html.parser")

        if not soup.title or not soup.title.string:
            empty_streak += 1
            continue

        name = soup.title.string.split(" - ")[0].strip()

        if name == "Fotbolltransfers.com":
            empty_streak += 1
            continue

        link = soup.find("link", rel="canonical")
        full_url = link["href"]
        full_id = full_url.replace("https://fotbolltransfers.com/", "")

        slug = slugify(name)

        resultsTeam.append({
            "id": full_id,
            "name": name,
            "url": full_url,
            "numeric_id": team_id
        })

        empty_streak = 0

    except requests.exceptions.RequestException:
        empty_streak += 1
        continue

    if team_id % SAVE_EVERY == 0:
        with open(SAVE_FILE, "w", encoding="utf-8") as f:
            json.dump(resultsTeam, f, ensure_ascii=False)

    if empty_streak >= MAX_EMPTY_STREAK:
        break

    time.sleep(0.1)

with open(FINAL_FILE, "w", encoding="utf-8") as f:
    json.dump(resultsTeam, f, ensure_ascii=False, indent=2)

100%|█████████████████████████████████████| 6799/6799 [1:44:07<00:00,  1.09it/s]


In [6]:
import json

with open("resultsTeam_snapshot.json", "w", encoding="utf-8") as f:
    json.dump(resultsTeam, f, ensure_ascii=False, indent=2)

print("Snapshot sparad ✅")

Snapshot sparad ✅


In [25]:
#resultsPlayer

In [7]:
import pandas as pd

In [28]:
dfPlayers = pd.DataFrame(resultsPlayer,columns=['id','name','url','numeric_id'])

In [29]:
dfPlayers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14090 entries, 0 to 14089
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          14090 non-null  object
 1   name        14090 non-null  object
 2   url         14090 non-null  object
 3   numeric_id  14090 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 440.4+ KB


In [30]:
dfPlayers.head(10)

,id,name,url,numeric_id
0,spelare/kim-christensen/37,Kim Christensen,https://fotbolltransfers.com/spelare/kim-chris...,37
1,spelare/marcus-sandberg/38,Marcus Sandberg,https://fotbolltransfers.com/spelare/marcus-sa...,38
2,spelare/karl-svensson/39,Karl Svensson,https://fotbolltransfers.com/spelare/karl-sven...,39
3,spelare/nicklas-carlsson/40,Nicklas Carlsson,https://fotbolltransfers.com/spelare/nicklas-c...,40
4,spelare/petter-björlund/41,Petter Björlund,https://fotbolltransfers.com/spelare/petter-bj...,41
5,spelare/tuomo-turunen/42,Tuomo Turunen,https://fotbolltransfers.com/spelare/tuomo-tur...,42
6,spelare/adam-johansson/43,Adam Johansson,https://fotbolltransfers.com/spelare/adam-joha...,43
7,spelare/tobias-hysén/45,Tobias Hysén,https://fotbolltransfers.com/spelare/tobias-hy...,45
8,spelare/thomas-olsson/46,Thomas Olsson,https://fotbolltransfers.com/spelare/thomas-ol...,46
9,spelare/stefan-selakovic/47,Stefan Selakovic,https://fotbolltransfers.com/spelare/stefan-se...,47


In [31]:
dfPlayer.tail(10)

,id,name,url,numeric_id
14080,spelare/elton-levin/15105,Elton Levin,https://fotbolltransfers.com/spelare/elton-lev...,15105
14081,spelare/oliver-klang/15106,Oliver Klang,https://fotbolltransfers.com/spelare/oliver-kl...,15106
14082,spelare/oliver-schaaf/15107,Oliver Schaaf,https://fotbolltransfers.com/spelare/oliver-sc...,15107
14083,spelare/vilmer-johansson/15108,Vilmer Johansson,https://fotbolltransfers.com/spelare/vilmer-jo...,15108
14084,spelare/mario-palomino-delgado/15109,Mario Palomino Delgado,https://fotbolltransfers.com/spelare/mario-pal...,15109
14085,spelare/sindri-kristinn-ólafsson/15110,Sindri Kristinn Ólafsson,https://fotbolltransfers.com/spelare/sindri-kr...,15110
14086,spelare/elion-lipovica/15111,Elion Lipovica,https://fotbolltransfers.com/spelare/elion-lip...,15111
14087,spelare/ebbe-orrling/15112,Ebbe Orrling,https://fotbolltransfers.com/spelare/ebbe-orrl...,15112
14088,spelare/hannes-nyman/15113,Hannes Nyman,https://fotbolltransfers.com/spelare/hannes-ny...,15113
14089,spelare/kota-sakurai/15114,Kota Sakurai,https://fotbolltransfers.com/spelare/kota-saku...,15114


In [32]:
resultsTeam

[{'id': 'klubbar/sverige/ifk-goteborg/2',
  'name': 'IFK Göteborg',
  'url': 'https://fotbolltransfers.com/klubbar/sverige/ifk-goteborg/2',
  'numeric_id': 2},
 {'id': 'klubbar/sverige/if-elfsborg/3',
  'name': 'IF Elfsborg',
  'url': 'https://fotbolltransfers.com/klubbar/sverige/if-elfsborg/3',
  'numeric_id': 3},
 {'id': 'klubbar/sverige/aik/4',
  'name': 'AIK',
  'url': 'https://fotbolltransfers.com/klubbar/sverige/aik/4',
  'numeric_id': 4},
 {'id': 'klubbar/sverige/assyriska-ff/6',
  'name': 'Assyriska FF',
  'url': 'https://fotbolltransfers.com/klubbar/sverige/assyriska-ff/6',
  'numeric_id': 6},
 {'id': 'klubbar/sverige/osters-if/7',
  'name': 'Östers IF',
  'url': 'https://fotbolltransfers.com/klubbar/sverige/osters-if/7',
  'numeric_id': 7},
 {'id': 'spelare/utlandsproffs/norge/8',
  'name': 'Svenska spelare i Norge',
  'url': 'https://fotbolltransfers.com/spelare/utlandsproffs/norge/8',
  'numeric_id': 8},
 {'id': 'klubbar/sverige/hammarby-if/9',
  'name': 'Hammarby IF',
  'u

In [33]:
dfTeam = pd.DataFrame(resultsTeam,columns=['id','name','url','numeric_id'])

In [34]:
dfTeam

,id,name,url,numeric_id
0,klubbar/sverige/ifk-goteborg/2,IFK Göteborg,https://fotbolltransfers.com/klubbar/sverige/i...,2
1,klubbar/sverige/if-elfsborg/3,IF Elfsborg,https://fotbolltransfers.com/klubbar/sverige/i...,3
2,klubbar/sverige/aik/4,AIK,https://fotbolltransfers.com/klubbar/sverige/a...,4
3,klubbar/sverige/assyriska-ff/6,Assyriska FF,https://fotbolltransfers.com/klubbar/sverige/a...,6
4,klubbar/sverige/osters-if/7,Östers IF,https://fotbolltransfers.com/klubbar/sverige/o...,7
...,...,...,...,...
6687,klubbar/ovriga-europa/mexico-fc/6795,México FC,https://fotbolltransfers.com/klubbar/ovriga-eu...,6795
6688,klubbar/sverige/fk-karlskrona-u16/6796,FK Karlskrona U16,https://fotbolltransfers.com/klubbar/sverige/f...,6796
6689,klubbar/ovriga-europa/rodina-moskva/6797,Rodina Moskva,https://fotbolltransfers.com/klubbar/ovriga-eu...,6797
6690,klubbar/ovriga-europa/fc-inter-u23/6798,FC Inter U23,https://fotbolltransfers.com/klubbar/ovriga-eu...,6798


### MixandMatch  
id,name,desc,type,url

In [35]:
dfTeam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6692 entries, 0 to 6691
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          6692 non-null   object
 1   name        6692 non-null   object
 2   url         6692 non-null   object
 3   numeric_id  6692 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 209.3+ KB


In [36]:
import pandas as pd
from datetime import datetime

# ===== KONFIG =====
TYPE_PLAYER = "Q5"        # människa
TYPE_TEAM = "Q476028"     # fotbollsklubb

# ===== FILNAMN =====
today = datetime.now().strftime("%Y%m%d")
filename = f"41_fotbolltransfers_mixnmatch_{today}.csv"

# ===== BYGG DATA =====

# Spelare
df_players_out = dfPlayers.copy()
df_players_out["desc"] = "fotbollsspelare"
df_players_out["type"] = TYPE_PLAYER

# Klubb
df_team_out = dfTeam.copy()
df_team_out["desc"] = "fotbollsklubb"
df_team_out["type"] = TYPE_TEAM

# ===== KOMBINERA =====
df_out = pd.concat([df_players_out, df_team_out], ignore_index=True)

# ===== VÄLJ KOLUMNER =====
df_out = df_out[["id", "name", "desc", "type", "url"]]

# ===== EXPORT =====
df_out.to_csv(filename, index=False, encoding="utf-8")

print(f"Fil skapad: {filename}")

Fil skapad: 41_fotbolltransfers_mixnmatch_20260323.csv


In [14]:
 # End timer and calculate duration
end_time = time.time()
elapsed_time = end_time - start_time# Bygg audit-lager för den här etappen

# Print current date and total time
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
minutes, seconds = divmod(elapsed_time, 60)
print("Total time elapsed: {:02.0f} minutes {:05.2f} seconds".format(minutes, seconds))


Date: 2026-03-20 21:03:15
Total time elapsed: 413 minutes 17.43 seconds
